# Image Upscaler Web App (Real-ESRGAN + Gradio)

Wraps Real-ESRGAN in a small Gradio web app: drop in an image, choose a scale, and get a sharper, larger version back. Gradio prints a public link you can open from any device while the notebook is running.

## How to run
- Open in Google Colab. Runtime type: **T4 GPU**.
- Run the cells from top to bottom.

_Rewritten from my original Colab notebook: `upscale.ipynb`._

In [ ]:
!pip install -q gradio

In [ ]:
!pip install -q realesrgan

## Fix a torchvision compatibility issue
`basicsr` (used by Real-ESRGAN) still imports `torchvision.transforms.functional_tensor`, which newer torchvision removed. This cell patches that one import line.

In [ ]:
import importlib.util
from pathlib import Path

# find_spec locates the package without importing it (importing would fail before the patch)
basicsr_dir = Path(importlib.util.find_spec("basicsr").origin).parent
deg_file = basicsr_dir / "data" / "degradations.py"
code = deg_file.read_text()
code = code.replace("torchvision.transforms.functional_tensor", "torchvision.transforms.functional")
deg_file.write_text(code)
print("Patched:", deg_file)

## Load the Real-ESRGAN model
The weights download automatically the first time.

In [ ]:
import torch
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer

# RealESRGAN_x4plus: a general-purpose 4x upscaling model
model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
upsampler = RealESRGANer(
    scale=4,
    model_path="https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
    model=model,
    tile=400,          # process the image in tiles so big images fit in GPU memory
    tile_pad=10,
    pre_pad=0,
    half=torch.cuda.is_available(),  # half precision is faster on GPU
)
print("Model ready")

In [ ]:
import numpy as np
from PIL import Image

def upscale(pil_image, outscale=4):
    """Upscale a PIL image with Real-ESRGAN and return a PIL image."""
    rgb = np.array(pil_image.convert("RGB"))
    bgr = rgb[:, :, ::-1]                      # Real-ESRGAN expects BGR (OpenCV order)
    out_bgr, _ = upsampler.enhance(bgr, outscale=outscale)
    return Image.fromarray(out_bgr[:, :, ::-1])

## Launch the app

In [ ]:
import gradio as gr

def app(image, scale):
    if image is None:
        return None
    return upscale(image, outscale=int(scale))

gr.Interface(fn=app,
             inputs=[gr.Image(type="pil", label="Low-resolution image"),
                     gr.Radio([2, 3, 4], value=4, label="Scale")],
             outputs=gr.Image(type="pil", label="Upscaled image", format="png"),
             title="Real-ESRGAN Image Upscaler").launch(share=True)